# Capstone Project-Prashanth A

### Import the basic Libs and  Data Injestion

In [1]:
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import datetime

# Define timeframe
start_date = "2005-01-01"
end_date = "2026-06-30"

# Define Asset Dictionaries
equities = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM', 'BRK-B', 'JNJ', 'XOM', 'PG', 'CAT', 'NEE', 'PLD']
fixed_income = ['TLT', 'IEF', 'LQD', 'HYG']
commodities = ['GLD', 'USO', 'DBA']
fx_crypto = ['EURUSD=X', 'USDJPY=X', 'BTC-USD', 'ETH-USD']
macro_features = ['BAMLC0A0CM', 'BAMLC0A4CBBB', 'BAMLH0A0HYM2', 'TB3MS', 'DGS2', 'DGS5', 'DGS10', 'DGS30', 'CPIAUCSL', 'UNRATE', 'INDPRO']

def fetch_yfinance_data(tickers):
    df = yf.download(tickers, start=start_date, end=end_date)
    return df.ffill().dropna()

def fetch_fred_data(tickers):
    df = web.DataReader(tickers, 'fred', start_date, end_date)
    return df.ffill().dropna()

print("Downloading Data...")
eq_df = fetch_yfinance_data(equities)
fi_df = fetch_yfinance_data(fixed_income)
cmdty_df = fetch_yfinance_data(commodities)
fx_cryp_df = fetch_yfinance_data(fx_crypto)
vix_df = fetch_yfinance_data(['^VIX'])
macro_df = fetch_fred_data(macro_features)

# --- FIX APPLIED HERE ---
# Extract only the 'Close' prices. This drops the top level of the MultiIndex,
# leaving a 1-level DataFrame with '^VIX' as the column name.
vix_close_df = vix_df['Close']

# Combine VIX with Macro for the overlays sheet
overlays_df = macro_df.join(vix_close_df, how='outer').ffill().dropna()

# Export to Multi-Sheet Excel
output_file = "IBAU25_Capstone_Data.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    eq_df.to_excel(writer, sheet_name='Equities_Data')
    fi_df.to_excel(writer, sheet_name='Fixed_Income_Data')
    cmdty_df.to_excel(writer, sheet_name='Commodities_Data')
    fx_cryp_df.to_excel(writer, sheet_name='FX_and_Crypto_Data')
    overlays_df.to_excel(writer, sheet_name='Macro_and_Credit_Overlays')

print(f"Data successfully exported to {output_file} with multiple sheets.")

[*********************100%***********************]  12 of 12 completed
[*********************100%***********************]  4 of 4 completed
[*********************100%***********************]  3 of 3 completed
[*********************100%***********************]  4 of 4 completed
[*********************100%***********************]  1 of 1 completed


Data successfully exported to IBAU25_Capstone_Data.xlsx with multiple sheets.


### Understanding the Basic Data

In [2]:
# Create a dictionary of your DataFrames for easy looping
dataframes = {
    "Equities (eq_df)": eq_df,
    "Fixed Income (fi_df)": fi_df,
    "Commodities (cmdty_df)": cmdty_df,
    "FX & Crypto (fx_cryp_df)": fx_cryp_df,
    "VIX (vix_df)": vix_df,
    "Macro (macro_df)": macro_df,
    "Overlays (overlays_df)": overlays_df
}

# Loop through and print the stats for each
for name, df in dataframes.items():
    print(f"====== {name} ======")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    
    print("--- HEAD (First 5 Rows) ---")
    print(df.head(), "\n")
    
    print("--- TAIL (Last 5 Rows) ---")
    print(df.tail(), "\n")
    print("="*60, "\n")

====== Equities (eq_df) ======
Shape: 5405 rows, 60 columns

--- HEAD (First 5 Rows) ---
Price          Close                                                      \
Ticker          AAPL    AMZN      BRK-B        CAT        JNJ        JPM   
Date                                                                       
2005-01-03  0.947307  2.2260  57.980000  28.122849  34.153622  22.055761   
2005-01-04  0.957036  2.1070  57.099998  27.812243  34.045021  21.828424   
2005-01-05  0.965418  2.0885  57.200001  27.279776  34.023308  21.873901   
2005-01-06  0.966166  2.0525  57.480000  27.682083  34.121037  21.998924   
2005-01-07  1.036514  2.1160  58.380001  27.622925  33.996151  21.822750   

Price                                                 ...   Volume           \
Ticker           MSFT       NEE      NVDA         PG  ...    BRK-B      CAT   
Date                                                  ...                     
2005-01-03  18.338739  4.864192  0.179939  30.441525  ...   69000